# AGORA z0 Results Viewer Only

This notebook only reads existing outputs under `parallel_outputs`. It does not import or run yt datasets and does not rerun LOS calculations.

In [ ]:
from pathlib import Path
import json
import csv
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

ROOT = Path('/home/zhaozhang/local/AGORA_work/AGORA_Data/z0')
OUTROOT = ROOT / 'parallel_outputs'
SELECTED_CODE = 'Enzo'  # change this for single-code plots
SELECTED_OUTDIR = OUTROOT / SELECTED_CODE

print('OUTROOT =', OUTROOT)
print('available outputs:')
for p in sorted(OUTROOT.glob('*')):
    if p.is_dir():
        print(' ', p.name)

FIGURE_DIR = OUTROOT / 'viewer_figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def safe_filename(name):
    return ''.join(c if c.isalnum() or c in '._-' else '_' for c in str(name)).strip('_')

def save_figure(fig, name, dpi=220):
    path = FIGURE_DIR / f'{safe_filename(name)}.png'
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    print('Saved figure:', path)
    return path


# By default, keep only production output directories in comparison plots.
# Smoke/debug/fix directories are useful for testing but should not enter science figures.
EXCLUDE_OUTPUT_NAME_PARTS = tuple(globals().get('EXCLUDE_OUTPUT_NAME_PARTS', ('_smoke', '_debug', '_fix')))

def is_production_output_name(name):
    return not any(part in str(name) for part in EXCLUDE_OUTPUT_NAME_PARTS)

def is_production_output_dir(path):
    path = Path(path)
    return path.is_dir() and is_production_output_name(path.name)

def load_json(path):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else None


def _find_nested_value(obj, names):
    if isinstance(obj, dict):
        for key, val in obj.items():
            if key in names:
                return val
            found = _find_nested_value(val, names)
            if found is not None:
                return found
    elif isinstance(obj, (list, tuple)):
        for val in obj:
            found = _find_nested_value(val, names)
            if found is not None:
                return found
    return None

def code_redshift(code, outroot=OUTROOT):
    code = str(code)
    for fname in ['parallel_pipeline_metadata.json', 'MWlike_4pi_DM_EM_metadata.json', 'projection_maps_face_edge_metadata.json']:
        meta = load_json(Path(outroot) / code / fname)
        if not meta:
            continue
        z = _find_nested_value(meta, {'current_redshift', 'redshift', 'Redshift', 'z'})
        if z is not None:
            try:
                z = float(z)
                if np.isfinite(z):
                    return z
            except Exception:
                pass
    # Some old output metadata was written before dataset_units were recorded.
    fallback = {'GADGET3': 0.3, 'GEAR': 0.2971321854409694}
    return fallback.get(code)

def code_label_with_redshift(code, z_threshold=0.05):
    z = code_redshift(code)
    if z is not None and abs(z) >= z_threshold:
        return f'{code}\n(z={z:.2f})'
    return str(code)



## Summary Table Across Codes

In [ ]:
def load_json(path):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else None

summary_rows = []
for outdir in sorted(OUTROOT.glob('*')):
    if not is_production_output_dir(outdir):
        continue
    summary = load_json(outdir / 'MWlike_4pi_DM_EM_summary.json')
    metadata = load_json(outdir / 'parallel_pipeline_metadata.json') or load_json(outdir / 'MWlike_4pi_DM_EM_metadata.json')
    if not summary:
        continue
    row = {'code': outdir.name}
    if metadata:
        cfg = metadata.get('config', {})
        row.update({
            'n_los': cfg.get('n_los'),
            'backend': metadata.get('integration_backend'),
            'ionization_mode': cfg.get('ionization_mode'),
            'center': metadata.get('center_kpc_original_frame'),
        })
    for key in ['DM_total_pc_cm3', 'EM_ne2_total_pc_cm6', 'EM_ne2_hot_pc_cm6']:
        if key in summary:
            row[f'{key}_median'] = summary[key].get('median')
            row[f'{key}_p16'] = summary[key].get('p16')
            row[f'{key}_p84'] = summary[key].get('p84')
    summary_rows.append(row)

try:
    import pandas as pd
    summary_table = pd.DataFrame(summary_rows)
    display(summary_table)
except Exception:
    for row in summary_rows:
        print(row)

In [ ]:


if summary_rows:
    codes = [r['code'] for r in summary_rows]
    panels = [
        ('DM_total_pc_cm3', r'DM [pc cm$^{-3}$]', 'tab:blue'),
        ('EM_ne2_total_pc_cm6', r'EM [pc cm$^{-6}$]', 'tab:red'),
        ('EM_ne2_hot_pc_cm6', r'Hot EM [pc cm$^{-6}$]', 'tab:orange'),
    ]
    fig, axes = plt.subplots(len(panels), 1, figsize=(10, 8.5), constrained_layout=True)
    for ax, (key, ylabel, color) in zip(axes, panels):
        med = np.array([r.get(f'{key}_median', np.nan) for r in summary_rows], dtype=float)
        lo = np.array([r.get(f'{key}_p16', np.nan) for r in summary_rows], dtype=float)
        hi = np.array([r.get(f'{key}_p84', np.nan) for r in summary_rows], dtype=float)
        yerr = np.vstack([med - lo, hi - med])
        ax.errorbar(codes, med, yerr=yerr, fmt='o', capsize=4, color=color)
        ax.set_yscale('log')
        ax.set_ylabel(ylabel)
        ax.grid(True, which='both', alpha=0.3)
        ax.tick_params(axis='x', rotation=10)
    save_figure(fig, f'comparison_DM_EM_hotEM_median_p16_p84.png')
    plt.show()

In [ ]:
SELECTED_CODE

## Projection Maps For One Code

In [ ]:
PLOT_CODE = SELECTED_CODE  # Change this to plot another code, e.g. 'GEAR', 'AREPO', 'G4Cal_Pablo'
PLOT_OUTDIR = OUTROOT / PLOT_CODE

font = {"family": "serif", "size": 16}
font_label = {"family": "serif", "size": 20}
tick_size = 18
font_legend = {"family": "serif", "size": 16}
font_legend_small = {"family": "serif", "size": 13}
PLOT_LIMIT_KPC = 15
FACE_EDGE_HEIGHT_RATIOS = [1.0, 0.45]
VELOCITY_QUIVER_STEP = 20
VELOCITY_QUIVER_SCALE = 120
VELOCITY_QUIVER_WIDTH = 0.004
VELOCITY_QUIVER_ALPHA = 0.70

plt.rcParams.update({"font.family": font["family"], "font.size": font["size"]})

def positive_lognorm(arr, pmin=2, pmax=98):
    arr = np.asarray(arr, dtype=float)
    pos = arr[np.isfinite(arr) & (arr > 0)]
    if pos.size == 0:
        return None
    vmin = max(np.nanpercentile(pos, pmin), 1e-30)
    vmax = max(np.nanpercentile(pos, pmax), vmin * 1.01)
    return LogNorm(vmin=vmin, vmax=vmax)

def paired_positive_norm(data, keys, pmin=2, pmax=98):
    vals = []
    for key in keys:
        if key in data:
            arr = np.asarray(data[key], dtype=float)
            pos = arr[np.isfinite(arr) & (arr > 0)]
            if pos.size:
                vals.append(pos)
    if not vals:
        return None
    vals = np.concatenate(vals)
    vmin = max(np.nanpercentile(vals, pmin), 1e-30)
    vmax = max(np.nanpercentile(vals, pmax), vmin * 1.01)
    return LogNorm(vmin=vmin, vmax=vmax)

def paired_symmetric_norm(data, keys, pmax=98):
    vals = []
    for key in keys:
        if key in data:
            arr = np.asarray(data[key], dtype=float)
            finite = arr[np.isfinite(arr)]
            if finite.size:
                vals.append(finite)
    if not vals:
        return None
    vals = np.concatenate(vals)
    vmax = np.nanpercentile(np.abs(vals), pmax)
    if not np.isfinite(vmax) or vmax <= 0:
        return None
    from matplotlib.colors import TwoSlopeNorm
    return TwoSlopeNorm(vcenter=0.0, vmin=-vmax, vmax=vmax)

def add_velocity_quiver(ax, data, view, step=VELOCITY_QUIVER_STEP):
    vx_key, vy_key, ext_key = f'gas_{view}_vx', f'gas_{view}_vy', f'gas_{view}_extent'
    if vx_key not in data or vy_key not in data or ext_key not in data:
        return
    vx = np.asarray(data[vx_key], dtype=float)
    vy = np.asarray(data[vy_key], dtype=float)
    if vx.ndim != 2 or vy.shape != vx.shape:
        return
    extent = np.asarray(data[ext_key], dtype=float)
    ny, nx = vx.shape
    xs = np.linspace(extent[0], extent[1], nx)
    ys = np.linspace(extent[2], extent[3], ny)
    ix = np.arange(0, nx, step)
    iy = np.arange(0, ny, step)
    X, Y = np.meshgrid(xs[ix], ys[iy])
    U = vx[np.ix_(iy, ix)]
    V = vy[np.ix_(iy, ix)]
    keep = (np.isfinite(U) & np.isfinite(V)
            & (X >= -PLOT_LIMIT_KPC) & (X <= PLOT_LIMIT_KPC)
            & (Y >= -PLOT_LIMIT_KPC) & (Y <= PLOT_LIMIT_KPC))
    if np.any(keep):
        ax.quiver(
            X[keep], Y[keep], U[keep], V[keep], color='black', angles='xy',
            scale_units='xy', scale=VELOCITY_QUIVER_SCALE,
            width=VELOCITY_QUIVER_WIDTH, alpha=VELOCITY_QUIVER_ALPHA,
            headwidth=4.6, headlength=5.8, headaxislength=5.0, minlength=0.08, zorder=6,
        )

def style_map_axis(ax, row, col):
    ax.set_xlim(-PLOT_LIMIT_KPC, PLOT_LIMIT_KPC)
    ax.set_ylim(-PLOT_LIMIT_KPC, PLOT_LIMIT_KPC)
    ax.set_aspect('auto')  # keep subplot rows aligned with the requested 1.7:1 height ratio
    ax.tick_params(axis='both', labelsize=tick_size)
    if row == 1:
        ax.set_xlabel('X [kpc]', fontdict=font_label)
    else:
        ax.set_xticklabels([])
    if col == 0:
        ax.set_ylabel('Y [kpc]', fontdict=font_label)
    else:
        ax.set_yticklabels([])

def annotate_view(ax, text):
    ax.text(
        0.03, 0.94, text, transform=ax.transAxes, ha='left', va='top',
        color='white', fontdict=font_legend_small,
        bbox=dict(facecolor='black', alpha=0.45, edgecolor='none', pad=2),
    )

proj_npz = PLOT_OUTDIR / 'projection_maps_face_edge.npz'
if proj_npz.exists():
    data = np.load(proj_npz)
    specs = [
        dict(title='Gas', face_key='gas_face_sigma', edge_key='gas_edge_sigma', face_extent='gas_face_extent', edge_extent='gas_edge_extent', cmap='magma', norm_func=paired_positive_norm, cbar=r'$M_\odot$ kpc$^{-2}$'),
        dict(title='Stellar', face_key='star_face_sigma', edge_key='star_edge_sigma', face_extent='star_face_extent', edge_extent='star_edge_extent', cmap='inferno', norm_func=paired_positive_norm, cbar=r'$M_\odot$ kpc$^{-2}$'),
        dict(title='Velocity', face_key='gas_face_vlos', edge_key='gas_edge_vlos', face_extent='gas_face_extent', edge_extent='gas_edge_extent', cmap='RdBu_r', norm_func=paired_symmetric_norm, cbar=r'km s$^{-1}$', quiver=True),
        dict(title='Temperature', face_key='gas_face_temperature', edge_key='gas_edge_temperature', face_extent='gas_face_extent', edge_extent='gas_edge_extent', cmap='plasma', norm_func=paired_positive_norm, cbar='K'),
    ]
    fig, axes = plt.subplots(
        2, len(specs), figsize=(5.0 * len(specs), 8.2),
        gridspec_kw={'height_ratios': FACE_EDGE_HEIGHT_RATIOS},
        constrained_layout=True, squeeze=False,
    )
    for col, spec in enumerate(specs):
        norm = spec['norm_func'](data, [spec['face_key'], spec['edge_key']])
        last_im = None
        for row, (arr_key, ext_key, view) in enumerate([
            (spec['face_key'], spec['face_extent'], 'face-on'),
            (spec['edge_key'], spec['edge_extent'], 'edge-on'),
        ]):
            ax = axes[row, col]
            if arr_key not in data or ext_key not in data:
                ax.set_axis_off()
                continue
            last_im = ax.imshow(data[arr_key], origin='lower', extent=data[ext_key], cmap=spec['cmap'], norm=norm, interpolation='nearest')
            style_map_axis(ax, row, col)
            annotate_view(ax, view)
            if spec.get('quiver'):
                add_velocity_quiver(ax, data, 'face' if row == 0 else 'edge')
        if last_im is not None:
            cbar = fig.colorbar(last_im, ax=axes[:, col], shrink=0.88, pad=0.015)
            cbar.ax.tick_params(labelsize=font_legend_small['size'])
            cbar.set_label(f"{spec['title']} [{spec['cbar']}]", fontdict=font_legend)
    fig.suptitle(f'{code_label_with_redshift(PLOT_CODE)}: Projection maps', fontdict={'family': 'serif', 'size': 24}, y=0.995)
    save_figure(fig, f'{PLOT_CODE}_projection_one_code')
    plt.show()
else:
    print('Missing:', proj_npz)


## Projection Maps Across All Codes


In [ ]:
# Across-code projection maps with selectable code list.
# Use "default"/"all"/None to plot every available valid code, or provide a list.
PLOT_CODE_LIST = globals().get('PLOT_CODE_LIST', 'default')
# Example:
PLOT_CODE_LIST = ['ARTI', 'AREPO', 'G4Cal_Pablo', 'GADGET3', 'GEAR', "ENZO"]#, "CHANGA"]

def _load_projection_products(outroot=OUTROOT):
    products = {}
    for outdir in sorted(Path(outroot).glob('*')):
        if not is_production_output_dir(outdir):
            continue
        path = outdir / 'projection_maps_face_edge.npz'
        if path.exists():
            try:
                products[outdir.name] = np.load(path)
            except Exception as exc:
                print('Cannot load', path, repr(exc))
    return products

def _code_selection_or_all(products, selection=None):
    available = list(products.keys())
    lookup = {str(code).lower(): code for code in available}
    if selection is None:
        selection = globals().get('PLOT_CODE_LIST', 'default')
    if selection is None or selection == 'default' or selection == 'all':
        selected = available
    elif isinstance(selection, str):
        selected = [selection]
    else:
        selected = list(selection)
    selected = [lookup.get(str(code).lower(), code) for code in selected]
    selected = [code for code in selected if code in products]
    valid = []
    for code in selected:
        meta = load_json(OUTROOT / code / 'parallel_pipeline_metadata.json') or load_json(OUTROOT / code / 'MWlike_4pi_DM_EM_metadata.json')
        cfg = meta.get('config', {}) if meta else {}
        if code == 'CHANGA' and cfg.get('tipsy_length_unit_kpc') is None:
            print('Skipping CHANGA: Tipsy physical units are missing.')
            continue
        valid.append(code)
    missing = sorted(set(selected) - set(valid) - ({'CHANGA'} if 'CHANGA' in selected else set()))
    if missing:
        print('Requested codes missing projection products:', missing)
    return valid

def _global_positive_norm(products, keys, pmin=2, pmax=98):
    vals = []
    for data in products.values():
        for key in keys:
            if key in data:
                arr = np.asarray(data[key], dtype=float)
                pos = arr[np.isfinite(arr) & (arr > 0)]
                if pos.size:
                    vals.append(pos)
    if not vals:
        return None
    vals = np.concatenate(vals)
    vmin = max(np.nanpercentile(vals, pmin), 1e-30)
    vmax = max(np.nanpercentile(vals, pmax), vmin * 1.01)
    return LogNorm(vmin=vmin, vmax=vmax)

def _global_symmetric_norm(products, keys, pmax=98):
    vals = []
    for data in products.values():
        for key in keys:
            if key in data:
                arr = np.asarray(data[key], dtype=float)
                finite = arr[np.isfinite(arr)]
                if finite.size:
                    vals.append(finite)
    if not vals:
        return None
    vals = np.concatenate(vals)
    vmax = np.nanpercentile(np.abs(vals), pmax)
    if not np.isfinite(vmax) or vmax <= 0:
        return None
    from matplotlib.colors import TwoSlopeNorm
    return TwoSlopeNorm(vcenter=0.0, vmin=-vmax, vmax=vmax)

def _style_overview_axis(ax, row, col, n_rows, n_cols, view):
    ax.set_xlim(-PLOT_LIMIT_KPC, PLOT_LIMIT_KPC)
    ax.set_ylim(-PLOT_LIMIT_KPC, PLOT_LIMIT_KPC)
    if view == 'face-on':
        ax.set_aspect('equal', adjustable='box')
    else:
        ax.set_aspect('auto')
    ax.tick_params(axis='both', labelsize=tick_size)
    if row == n_rows - 1:
        ax.set_xlabel('X [kpc]', fontdict=font_label)
    else:
        ax.set_xticklabels([])
    if col == 0:
        ax.set_ylabel('Y [kpc]', fontdict=font_label)
    else:
        ax.set_yticklabels([])

def _projection_group_specs(products):
    return [
        dict(label=r'$\Sigma_{\rm gas}$', title='Gas', face_key='gas_face_sigma', edge_key='gas_edge_sigma', face_extent='gas_face_extent', edge_extent='gas_edge_extent', cmap='magma', norm=_global_positive_norm(products, ['gas_face_sigma', 'gas_edge_sigma']), cbar=r'$M_\odot$ kpc$^{-2}$'),
        dict(label=r'$\Sigma_{\star}$', title='Stellar', face_key='star_face_sigma', edge_key='star_edge_sigma', face_extent='star_face_extent', edge_extent='star_edge_extent', cmap='inferno', norm=_global_positive_norm(products, ['star_face_sigma', 'star_edge_sigma']), cbar=r'$M_\odot$ kpc$^{-2}$'),
        dict(label='Temperature', title='Temperature', face_key='gas_face_temperature', edge_key='gas_edge_temperature', face_extent='gas_face_extent', edge_extent='gas_edge_extent', cmap='plasma', norm=_global_positive_norm(products, ['gas_face_temperature', 'gas_edge_temperature']), cbar='K'),
        dict(label='Velocity', title='Velocity', face_key='gas_face_vlos', edge_key='gas_edge_vlos', face_extent='gas_face_extent', edge_extent='gas_edge_extent', cmap='RdBu_r', norm=_global_symmetric_norm(products, ['gas_face_vlos', 'gas_edge_vlos']), cbar=r'km s$^{-1}$', quiver=True),
    ]

def _plot_one_quantity_block(products, codes, spec):
    fig, axes = plt.subplots(
        2, len(codes), figsize=(3.35 * len(codes), 7.0),
        gridspec_kw={'height_ratios': FACE_EDGE_HEIGHT_RATIOS},
        constrained_layout=True, squeeze=False,
    )
    last_im = None
    for col, code in enumerate(codes):
        data = products[code]
        for row, (arr_key, ext_key, view) in enumerate([
            (spec['face_key'], spec['face_extent'], 'face-on'),
            (spec['edge_key'], spec['edge_extent'], 'edge-on'),
        ]):
            ax = axes[row, col]
            if arr_key not in data or ext_key not in data:
                ax.set_axis_off()
                continue
            last_im = ax.imshow(data[arr_key], origin='lower', extent=data[ext_key], cmap=spec['cmap'], norm=spec['norm'], interpolation='nearest')
            _style_overview_axis(ax, row, col, 2, len(codes), view)
            annotate_view(ax, view)
            if spec.get('quiver'):
                add_velocity_quiver(ax, data, 'face' if row == 0 else 'edge')
        axes[0, col].set_title(code_label_with_redshift(code), fontdict=font_label)
    if last_im is not None:
        cbar = fig.colorbar(last_im, ax=axes.ravel().tolist(), shrink=0.88, pad=0.01)
        cbar.set_label(f"{spec['label']} [{spec['cbar']}]", fontdict=font_legend)
        cbar.ax.tick_params(labelsize=font_legend_small['size'])
    fig.suptitle('Projection maps across codes', fontdict={'family': 'serif', 'size': 24})
    save_figure(fig, f"across_codes_{spec['title']}_projection")
    plt.show()

projection_products = _load_projection_products()
valid_codes = _code_selection_or_all(projection_products, PLOT_CODE_LIST)
if not valid_codes:
    print('No selected projection products found under', OUTROOT)
else:
    products = {code: projection_products[code] for code in valid_codes}
    for spec in _projection_group_specs(products):
        _plot_one_quantity_block(products, valid_codes, spec)


## Nested Projection Atlas Across Codes


In [ ]:
# Compact nested atlas: columns are codes, rows are physical quantities.
# Each cell contains face-on on top and edge-on below.

projection_products = _load_projection_products()
atlas_codes = _code_selection_or_all(projection_products, PLOT_CODE_LIST)
if not atlas_codes:
    print('No selected projection products found under', OUTROOT)
else:
    atlas_products = {code: projection_products[code] for code in atlas_codes}
    atlas_specs = _projection_group_specs(atlas_products)
    n_rows = len(atlas_specs)
    n_cols = len(atlas_codes)

    fig = plt.figure(figsize=(3.15 * n_cols + 0.8, 4.75 * n_rows), constrained_layout=False)
    outer = fig.add_gridspec(
        n_rows, n_cols + 1,
        width_ratios=[1.0] * n_cols + [0.055],
        height_ratios=[1.0] * n_rows,
        left=0.085, right=0.945, bottom=0.06, top=0.935,
        wspace=0.055, hspace=0.18,
    )

    def _format_nested_axis(ax, row, col, i, n_rows, n_cols, view):
        ax.set_xlim(-PLOT_LIMIT_KPC, PLOT_LIMIT_KPC)
        ax.set_ylim(-PLOT_LIMIT_KPC, PLOT_LIMIT_KPC)
        if view == 'face-on':
            ax.set_box_aspect(1.0)
            ax.set_aspect('equal', adjustable='box')
        else:
            ax.set_box_aspect(0.45)
            ax.set_aspect('auto')
        ax.tick_params(axis='both', labelsize=tick_size, direction='out', pad=2)
        if not (row == n_rows - 1 and i == 1):
            ax.tick_params(labelbottom=False)
            ax.set_xlabel('')
        else:
            ax.set_xlabel('X [kpc]', fontdict=font_label, labelpad=3)
        if col != 0:
            ax.tick_params(labelleft=False)
            ax.set_ylabel('')
        else:
            ax.set_ylabel('Y [kpc]', fontdict=font_label, labelpad=3)

    for row, spec in enumerate(atlas_specs):
        row_axes = []
        last_im = None
        for col, code in enumerate(atlas_codes):
            data = atlas_products[code]
            sub = outer[row, col].subgridspec(
                2, 1,
                height_ratios=[1.0, 0.45],
                hspace=0.035,
            )
            face_ax = fig.add_subplot(sub[0, 0])
            edge_ax = fig.add_subplot(sub[1, 0])
            row_axes.extend([face_ax, edge_ax])
            for i, (ax, arr_key, ext_key, view) in enumerate([
                (face_ax, spec['face_key'], spec['face_extent'], 'face-on'),
                (edge_ax, spec['edge_key'], spec['edge_extent'], 'edge-on'),
            ]):
                if arr_key not in data or ext_key not in data:
                    ax.set_axis_off()
                    continue
                last_im = ax.imshow(
                    data[arr_key], origin='lower', extent=data[ext_key], cmap=spec['cmap'],
                    norm=spec['norm'], interpolation='nearest'
                )
                _format_nested_axis(ax, row, col, i, n_rows, n_cols, view)
                annotate_view(ax, view)
                if spec.get('quiver'):
                    add_velocity_quiver(ax, data, 'face' if i == 0 else 'edge')
            if row == 0:
                face_ax.set_title(code_label_with_redshift(code), fontdict=font_label, pad=8)
        if last_im is not None:
            cax = fig.add_subplot(outer[row, -1])
            cbar = fig.colorbar(last_im, cax=cax)
            cbar.set_label(f"{spec['label']} [{spec['cbar']}]", fontdict=font_legend)
            cbar.ax.tick_params(labelsize=font_legend_small['size'])
    fig.suptitle('Projection atlas across codes', fontdict={'family': 'serif', 'size': 26}, y=0.985)
    save_figure(fig, 'across_codes_nested_projection_atlas')
    plt.show()


In [ ]:
OUTROOT / DIAGNOSTIC_CODE

## Mollweide And Hot-EM Diagnostic Panels


In [ ]:
# Existing diagnostic products for one selected code.
# Set DIAGNOSTIC_CODE_OVERRIDE = 'GADGET3' (or another code) to force a code.
# Leave it as None to follow PLOT_CODE.
DIAGNOSTIC_CODE_OVERRIDE = globals().get('DIAGNOSTIC_CODE_OVERRIDE', None)
DIAGNOSTIC_CODE = DIAGNOSTIC_CODE_OVERRIDE or PLOT_CODE
DIAGNOSTIC_OUTDIR = OUTROOT / DIAGNOSTIC_CODE
DIAGNOSTIC_LABEL = code_label_with_redshift(DIAGNOSTIC_CODE)

mollweide_files = [
    ('DM', 'MWlike_4pi_DM_total_pc_cm3_mollweide.png'),
    ('Total EM', 'MWlike_4pi_EM_ne2_total_pc_cm6_mollweide.png'),
    ('Hot EM', 'MWlike_4pi_EM_ne2_hot_pc_cm6_mollweide.png'),
]
existing = [(label, DIAGNOSTIC_OUTDIR / fname) for label, fname in mollweide_files if (DIAGNOSTIC_OUTDIR / fname).exists()]
if not existing:
    print('No Mollweide PNGs found for', DIAGNOSTIC_CODE, 'under', DIAGNOSTIC_OUTDIR)
else:
    fig, axes = plt.subplots(len(existing), 1, figsize=(13, 4.6 * len(existing)), constrained_layout=True)
    axes = np.atleast_1d(axes)
    for ax, (label, path) in zip(axes, existing):
        ax.imshow(plt.imread(path))
        ax.set_title(f'{DIAGNOSTIC_LABEL}: {label} Mollweide', fontdict=font_label)
        ax.set_axis_off()
    save_figure(fig, f'{DIAGNOSTIC_CODE}_mollweide_diagnostics')
    plt.show()

def _plot_hot_em_vs_gc_angle_logy(ax, outdir, code_label):
    sightline_path = Path(outdir) / 'MWlike_4pi_DM_EM_sightlines.npz'
    if not sightline_path.exists():
        ax.set_axis_off()
        ax.text(0.5, 0.5, f'Missing {sightline_path.name}', transform=ax.transAxes, ha='center', va='center')
        return False
    with np.load(sightline_path) as data:
        if 'angle_from_galactic_center_deg' not in data.files or 'EM_ne2_hot_pc_cm6' not in data.files:
            ax.set_axis_off()
            ax.text(0.5, 0.5, 'Missing angle or hot-EM arrays', transform=ax.transAxes, ha='center', va='center')
            return False
        angle = np.asarray(data['angle_from_galactic_center_deg'], dtype=float)
        hot_em = np.asarray(data['EM_ne2_hot_pc_cm6'], dtype=float)
    mask = np.isfinite(angle) & np.isfinite(hot_em) & (angle > 0) & (hot_em >= 0)
    angle = angle[mask]
    hot_em = hot_em[mask]
    if angle.size == 0:
        ax.set_axis_off()
        ax.text(0.5, 0.5, 'No positive-angle hot-EM samples', transform=ax.transAxes, ha='center', va='center')
        return False
    ax.scatter(angle, hot_em, s=9, alpha=0.45, color='#d9951e', edgecolors='none', label='LOS samples')
    ymin = max(np.nanmin(hot_em), 0.1)
    ymax = max(np.nanmax(hot_em), ymin * 1.1)
    bins = np.logspace(np.log10(ymin), np.log10(ymax), 28)
    centers = np.sqrt(bins[:-1] * bins[1:])
    med = np.full_like(centers, np.nan, dtype=float)
    p16 = np.full_like(centers, np.nan, dtype=float)
    p84 = np.full_like(centers, np.nan, dtype=float)
    for i in range(len(centers)):
        in_bin = (angle >= bins[i]) & (angle < bins[i + 1])
        if np.count_nonzero(in_bin) >= 5:
            med[i] = np.nanmedian(hot_em[in_bin])
            p16[i], p84[i] = np.nanpercentile(hot_em[in_bin], [16, 84])
    ok = np.isfinite(med)
    if np.any(ok):
        ax.fill_between(centers[ok], p16[ok], p84[ok], color='0.7', alpha=0.5, lw=0, label='16-84%')
        ax.plot(centers[ok], med[ok], color='black', lw=2.3, label='binned median')
    ax.set_yscale('log')
    ax.set_ylim(ymin, ymax)
    ax.set_xlabel('Angle from Galactic centre [deg]', fontdict=font_label)
    ax.set_ylabel(r'Hot EM [pc cm$^{-6}$]', fontdict=font_label)
    ax.set_title(f'{code_label}: Hot EM vs GC angle', fontdict=font_label)
    ax.tick_params(axis='both', labelsize=tick_size)
    ax.grid(True, which='both', alpha=0.25)
    ax.legend(frameon=True, fontsize=font_legend_small['size'])
    return True

polar_path = DIAGNOSTIC_OUTDIR / 'MWlike_4pi_hot_EM_GC_polar.png'
if not polar_path.exists() and not (DIAGNOSTIC_OUTDIR / 'MWlike_4pi_DM_EM_sightlines.npz').exists():
    print('No hot-EM diagnostic products found for', DIAGNOSTIC_CODE, 'under', DIAGNOSTIC_OUTDIR)
else:
    fig, axes = plt.subplots(1, 2, figsize=(14.5, 6.4), constrained_layout=True)
    if polar_path.exists():
        axes[0].imshow(plt.imread(polar_path))
        axes[0].set_title(f'{DIAGNOSTIC_LABEL}: Hot EM GC polar', fontdict=font_label)
        axes[0].set_axis_off()
    else:
        axes[0].set_axis_off()
        axes[0].text(0.5, 0.5, 'Missing polar PNG', transform=axes[0].transAxes, ha='center', va='center')
    _plot_hot_em_vs_gc_angle_logy(axes[1], DIAGNOSTIC_OUTDIR, DIAGNOSTIC_LABEL)
    save_figure(fig, f'{DIAGNOSTIC_CODE}_hot_em_diagnostics')
    plt.show()


## DM PDF Parametric Fits Across Codes


In [ ]:
# DM PDF data + parametric fits. This is intentionally DM-only for clarity.
from scipy import stats, special

def load_dm_fit_distributions(outroot=OUTROOT):
    per_code = {}
    for npz_path in sorted(Path(outroot).glob('*/MWlike_4pi_DM_EM_sightlines.npz')):
        code_name = npz_path.parent.name
        if not is_production_output_name(code_name):
            continue
        with np.load(npz_path) as data:
            if 'DM_total_pc_cm3' not in data.files:
                continue
            vals = np.asarray(data['DM_total_pc_cm3'], dtype=float)
            vals = vals[np.isfinite(vals) & (vals > 0)]
            if vals.size > 5:
                per_code[code_name] = {'DM_total_pc_cm3': vals}
    return per_code


def fit_log10_gmm(y, n_components=3, max_iter=300, tol=1e-7):
    """Fit a 1D Gaussian mixture to log10 values with a small EM implementation."""
    y = np.asarray(y, dtype=float)
    y = y[np.isfinite(y)]
    if y.size < max(20, 5 * n_components):
        raise ValueError('not enough samples for GMM')
    qs = np.linspace(10, 90, n_components)
    means = np.nanpercentile(y, qs)
    sigma0 = np.nanstd(y)
    if not np.isfinite(sigma0) or sigma0 <= 0:
        raise ValueError('zero scatter in log10 data')
    sigmas = np.full(n_components, max(sigma0 / n_components, 1e-3), dtype=float)
    weights = np.full(n_components, 1.0 / n_components, dtype=float)
    prev_ll = -np.inf
    for _ in range(max_iter):
        log_resp = np.column_stack([
            np.log(max(weights[k], 1e-300)) + stats.norm.logpdf(y, loc=means[k], scale=max(sigmas[k], 1e-6))
            for k in range(n_components)
        ])
        log_norm = special.logsumexp(log_resp, axis=1)
        ll = float(np.sum(log_norm))
        resp = np.exp(log_resp - log_norm[:, None])
        nk = resp.sum(axis=0) + 1e-300
        weights = nk / y.size
        means = (resp * y[:, None]).sum(axis=0) / nk
        var = (resp * (y[:, None] - means) ** 2).sum(axis=0) / nk
        sigmas = np.sqrt(np.maximum(var, 1e-6))
        if np.isfinite(prev_ll) and abs(ll - prev_ll) < tol * (abs(prev_ll) + 1.0):
            break
        prev_ll = ll
    order = np.argsort(means)
    return weights[order], means[order], sigmas[order], ll

def log10_gmm_pdf_x(x, weights, means, sigmas):
    """Convert a Gaussian mixture PDF in y=log10(x) to a PDF in x."""
    x = np.asarray(x, dtype=float)
    y = np.log10(x)
    pdf_y = np.zeros_like(x, dtype=float)
    for w, mu, sigma in zip(weights, means, sigmas):
        pdf_y += w * stats.norm.pdf(y, loc=mu, scale=max(sigma, 1e-6))
    return pdf_y / (x * np.log(10.0))

DM_FIT_CODE_LIST = globals().get('DM_FIT_CODE_LIST', PLOT_CODE_LIST)
all_dist = load_dm_fit_distributions()
if 'projection_products' in globals():
    fit_codes = _code_selection_or_all({code: None for code in all_dist}, DM_FIT_CODE_LIST)
else:
    fit_codes = list(all_dist)
fit_codes = [code for code in fit_codes if code in all_dist and 'DM_total_pc_cm3' in all_dist[code]]

if not fit_codes:
    print('No DM distributions found for selected codes.')
else:
    ncols = min(3, len(fit_codes))
    nrows = int(np.ceil(len(fit_codes) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.0 * ncols, 4.7 * nrows), constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()
    fit_rows = []
    for ax, code in zip(axes, fit_codes):
        vals = np.asarray(all_dist[code]['DM_total_pc_cm3'], dtype=float)
        vals = vals[np.isfinite(vals) & (vals > 0)]
        xmin = max(np.nanpercentile(vals, 0.2), np.nanmin(vals))
        xmax = np.nanpercentile(vals, 99.8)
        bins = np.logspace(np.log10(xmin), np.log10(xmax), 52)
        xgrid = np.logspace(np.log10(xmin), np.log10(xmax), 500)
        ax.hist(vals, bins=bins, density=True, histtype='step', lw=2.1, color='black', label=f'data (N={len(vals)})')

        try:
            shape, loc, scale = stats.lognorm.fit(vals, floc=0)
            ax.plot(xgrid, stats.lognorm.pdf(xgrid, shape, loc=loc, scale=scale), lw=2, label='Log-normal')
            fit_rows.append({'code': code, 'model': 'lognormal', 'shape': shape, 'loc': loc, 'scale': scale})
        except Exception as exc:
            print(code, 'lognormal fit failed:', repr(exc))
        try:
            weights, means, sigmas, loglike = fit_log10_gmm(np.log10(vals), n_components=3)
            gmm_pdf = log10_gmm_pdf_x(xgrid, weights, means, sigmas)
            ax.plot(xgrid, gmm_pdf, lw=2, ls='-.', label='GMM-3 log10')
            fit_rows.append({
                'code': code,
                'model': 'gmm3_log10',
                'weights': ';'.join(f'{v:.8g}' for v in weights),
                'means_log10': ';'.join(f'{v:.8g}' for v in means),
                'sigmas_log10': ';'.join(f'{v:.8g}' for v in sigmas),
                'loglike': loglike,
            })
        except Exception as exc:
            print(code, 'GMM-3 log10 fit failed:', repr(exc))
        try:
            c, loc, scale = stats.weibull_min.fit(vals, floc=0)
            ax.plot(xgrid, stats.weibull_min.pdf(xgrid, c, loc=loc, scale=scale), lw=2, ls='--', label='Weibull')
            fit_rows.append({'code': code, 'model': 'weibull', 'shape': c, 'loc': loc, 'scale': scale})
        except Exception as exc:
            print(code, 'weibull fit failed:', repr(exc))
        try:
            threshold = np.nanpercentile(vals, 90)
            tail = vals[vals >= threshold] - threshold
            gpd_c, gpd_loc, gpd_scale = stats.genpareto.fit(tail, floc=0)
            tail_pdf = np.zeros_like(xgrid)
            mask = xgrid >= threshold
            tail_frac = tail.size / vals.size
            tail_pdf[mask] = tail_frac * stats.genpareto.pdf(xgrid[mask] - threshold, gpd_c, loc=gpd_loc, scale=gpd_scale)
            ax.plot(xgrid[mask], tail_pdf[mask], lw=2, ls=':', label='GPD tail')
            fit_rows.append({'code': code, 'model': 'gpd_tail_p90', 'shape': gpd_c, 'loc': gpd_loc, 'scale': gpd_scale, 'threshold': threshold})
        except Exception as exc:
            print(code, 'GPD tail fit failed:', repr(exc))

        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_title(code, fontdict=font_label)
        ax.set_xlabel(r'DM [pc cm$^{-3}$]', fontdict=font_label)
        ax.set_ylabel('PDF', fontdict=font_label)
        ax.tick_params(axis='both', labelsize=tick_size)
        ax.grid(True, which='both', alpha=0.25)
        ax.legend(frameon=False, fontsize=font_legend_small['size'])
    for ax in axes[len(fit_codes):]:
        ax.set_axis_off()
    save_figure(fig, 'DM_pdf_parametric_fits')
    plt.show()

    if fit_rows:
        try:
            import pandas as pd
            fit_table = pd.DataFrame(fit_rows)
            fit_table.to_csv(FIGURE_DIR / 'DM_pdf_parametric_fit_parameters.csv', index=False)
            display(fit_table)
        except Exception:
            print(fit_rows)


## LOS PDF Histograms Across Codes

In [ ]:
FIT_QUANTITIES = {
    'DM_total_pc_cm3': r'DM [pc cm$^{-3}$]',
    'EM_ne2_total_pc_cm6': r'EM [pc cm$^{-6}$]',
    'EM_ne2_hot_pc_cm6': r'Hot EM [pc cm$^{-6}$]',
}
PDF_XMIN_BY_QUANTITY = {
    'DM_total_pc_cm3': 10,
    'EM_ne2_total_pc_cm6': 1e-2,
    'EM_ne2_hot_pc_cm6': 1e-2,
}
PDF_LEGEND_SIZE = max(8, font_legend_small['size'] - 2)

def load_per_code_distributions(outroot=OUTROOT):
    per_code = {}
    for npz_path in sorted(Path(outroot).glob('*/MWlike_4pi_DM_EM_sightlines.npz')):
        code_name = npz_path.parent.name
        if not is_production_output_name(code_name):
            continue
        with np.load(npz_path) as data:
            arrays = {}
            for key in FIT_QUANTITIES:
                if key in data.files:
                    vals = np.asarray(data[key], dtype=float)
                    vals = vals[np.isfinite(vals) & (vals > 0)]
                    if vals.size > 5:
                        arrays[key] = vals
        if arrays:
            per_code[code_name] = arrays
    return per_code

per_code = load_per_code_distributions()
quantities = [q for q in FIT_QUANTITIES if any(q in v for v in per_code.values())]
if quantities:
    colors = plt.cm.tab10(np.linspace(0, 1, max(3, len(per_code))))
    code_colors = {code: colors[i] for i, code in enumerate(sorted(per_code))}
    fig, axes = plt.subplots(len(quantities), 1, figsize=(10, 4.5 * len(quantities)), constrained_layout=True)
    axes = np.atleast_1d(axes)
    for ax, quantity in zip(axes, quantities):
        all_vals = np.concatenate([v[quantity] for v in per_code.values() if quantity in v])
        xmin = PDF_XMIN_BY_QUANTITY.get(quantity, 1e-2)
        all_vals = all_vals[np.isfinite(all_vals) & (all_vals >= xmin)]
        if all_vals.size == 0:
            ax.set_axis_off()
            ax.text(0.5, 0.5, f'No {quantity} values >= {xmin:g}', transform=ax.transAxes, ha='center', va='center')
            continue
        xmax = max(np.nanpercentile(all_vals, 99.8), xmin * 1.1)
        bins = np.logspace(np.log10(xmin), np.log10(xmax), 56)
        for code, arrays in sorted(per_code.items()):
            if quantity not in arrays:
                continue
            vals = arrays[quantity]
            vals = vals[np.isfinite(vals) & (vals >= xmin)]
            if vals.size == 0:
                continue
            ax.hist(
                vals, bins=bins, density=True, histtype='step', lw=2,
                color=code_colors[code], label=f'{code_label_with_redshift(code)} (N={len(vals)})'
            )
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_xlim(xmin, xmax)
        ax.set_xlabel(FIT_QUANTITIES[quantity], fontdict=font_label)
        ax.set_ylabel('PDF', fontdict=font_label)
        ax.tick_params(axis='both', labelsize=tick_size)
        ax.grid(True, which='both', alpha=0.25)
        ax.legend(ncol=2, frameon=False, fontsize=PDF_LEGEND_SIZE)
    save_figure(fig, 'LOS_pdf_histograms_across_codes')
    plt.show()
else:
    print('No LOS PDF quantities found under', OUTROOT)


## DM and EM Component Contributions


In [ ]:
# DM/EM component contribution distributions for one selected code.
# Change COMPONENT_CODE to inspect another simulation output.
COMPONENT_CODE = globals().get('COMPONENT_CODE', globals().get('PLOT_CODE', SELECTED_CODE))
COMPONENT_OUTDIR = OUTROOT / COMPONENT_CODE
component_npz = COMPONENT_OUTDIR / 'MWlike_4pi_DM_EM_sightlines.npz'

component_specs = [
    (
        'DM component distribution',
        r'DM [pc cm$^{-3}$]',
        [
            ('Total', 'DM_total_pc_cm3'),
            ('ISM', 'DM_ISM_pc_cm3'),
            ('CGM', 'DM_CGM_pc_cm3'),
            ('Hot', 'DM_hot_pc_cm3'),
        ],
    ),
    (
        'EM component distribution',
        r'EM [pc cm$^{-6}$]',
        [
            ('Total', 'EM_ne2_total_pc_cm6'),
            ('ISM', 'EM_ne2_ISM_pc_cm6'),
            ('CGM', 'EM_ne2_CGM_pc_cm6'),
            ('Hot', 'EM_ne2_hot_pc_cm6'),
        ],
    ),
]

def _component_array(data, key):
    if key not in data.files:
        return np.array([], dtype=float)
    vals = np.asarray(data[key], dtype=float)
    return vals[np.isfinite(vals) & (vals > 0)]

if not component_npz.exists():
    print('Missing:', component_npz)
else:
    with np.load(component_npz) as data:
        fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.2), constrained_layout=True)
        rng = np.random.default_rng(12345)
        for ax, (title, ylabel, entries) in zip(axes, component_specs):
            labels = []
            arrays = []
            for label, key in entries:
                vals = _component_array(data, key)
                if vals.size == 0:
                    print(f'{COMPONENT_CODE}: missing or empty {key}')
                labels.append(label)
                arrays.append(vals if vals.size else np.array([np.nan]))

            ax.boxplot(arrays, labels=labels, showfliers=False)
            for i, vals in enumerate(arrays, start=1):
                vals = vals[np.isfinite(vals) & (vals > 0)]
                if vals.size == 0:
                    continue
                jitter = rng.uniform(-0.08, 0.08, vals.size)
                ax.scatter(np.full(vals.size, i) + jitter, vals, s=7, alpha=0.35, rasterized=True)

            ax.set_yscale('log')
            ax.set_ylabel(ylabel, fontdict=font_label if 'font_label' in globals() else None)
            ax.set_title(title, fontdict=font_label if 'font_label' in globals() else None)
            ax.tick_params(axis='both', labelsize=tick_size if 'tick_size' in globals() else None)
            ax.grid(True, axis='y', which='both', alpha=0.28)
        fig.suptitle(f'{code_label_with_redshift(COMPONENT_CODE)}: DM/EM component contributions', fontdict={'family': 'serif', 'size': 22})
        save_figure(fig, f'{COMPONENT_CODE}_DM_EM_component_distributions')
        plt.show()


In [ ]:
# DM/EM component contribution distributions for all selected codes.
# Use PLOT_CODE_LIST, or set COMPONENT_CODE_LIST manually.  
#COMPONENT_CODE_LIST = ['ARTI', 'Enzo', 'G4Cal_Pablo', 'GADGET3', 'GEAR']
#COMPONENT_CODE_LIST = 'all'
COMPONENT_CODE_LIST = globals().get('COMPONENT_CODE_LIST', globals().get('PLOT_CODE_LIST', 'default'))

def _available_component_codes(outroot=OUTROOT):
    return sorted(p.parent.name for p in Path(outroot).glob('*/MWlike_4pi_DM_EM_sightlines.npz') if is_production_output_name(p.parent.name))

def _select_component_codes(selection=COMPONENT_CODE_LIST):
    available = _available_component_codes()
    lookup = {str(code).lower(): code for code in available}
    if selection is None or selection == 'default' or selection == 'all':
        selected = available
    elif isinstance(selection, str):
        selected = [selection]
    else:
        selected = list(selection)
    selected = [lookup.get(str(code).lower(), code) for code in selected]
    selected = [code for code in selected if code in available]
    if not selected:
        print('No selected component files found. Available:', available)
    return selected

component_specs = [
    (
        'DM component distribution',
        r'DM [pc cm$^{-3}$]',
        [
            ('Total', 'DM_total_pc_cm3'),
            ('ISM', 'DM_ISM_pc_cm3'),
            ('CGM', 'DM_CGM_pc_cm3'),
            ('Hot', 'DM_hot_pc_cm3'),
        ],
    ),
    (
        'EM component distribution',
        r'EM [pc cm$^{-6}$]',
        [
            ('Total', 'EM_ne2_total_pc_cm6'),
            ('ISM', 'EM_ne2_ISM_pc_cm6'),
            ('CGM', 'EM_ne2_CGM_pc_cm6'),
            ('Hot', 'EM_ne2_hot_pc_cm6'),
        ],
    ),
]

def _component_array(data, key):
    if key not in data.files:
        return np.array([], dtype=float)
    vals = np.asarray(data[key], dtype=float)
    return vals[np.isfinite(vals) & (vals > 0)]

component_codes = _select_component_codes()

for component_code in component_codes:
    component_npz = OUTROOT / component_code / 'MWlike_4pi_DM_EM_sightlines.npz'
    with np.load(component_npz) as data:
        fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.2), constrained_layout=True)
        rng = np.random.default_rng(12345)

        for ax, (title, ylabel, entries) in zip(axes, component_specs):
            labels = []
            arrays = []

            for label, key in entries:
                vals = _component_array(data, key)
                labels.append(label)
                arrays.append(vals if vals.size else np.array([np.nan]))

            ax.boxplot(arrays, labels=labels, showfliers=False)

            for i, vals in enumerate(arrays, start=1):
                vals = vals[np.isfinite(vals) & (vals > 0)]
                if vals.size == 0:
                    continue
                jitter = rng.uniform(-0.08, 0.08, vals.size)
                ax.scatter(
                    np.full(vals.size, i) + jitter,
                    vals,
                    s=7,
                    alpha=0.35,
                    rasterized=True,
                )

            ax.set_yscale('log')
            ax.set_ylabel(ylabel, fontdict=font_label)
            ax.set_title(title, fontdict=font_label)
            ax.tick_params(axis='both', labelsize=tick_size)
            ax.grid(True, axis='y', which='both', alpha=0.28)

        fig.suptitle(
            f'{code_label_with_redshift(component_code)}: DM/EM component contributions',
            fontdict={'family': 'serif', 'size': 22},
        )
        save_figure(fig, f'{component_code}_DM_EM_component_distributions')
        plt.show()

## Random Observer Special LOS

In [ ]:
# Plot random-observer special LOS results.
# Default follows the current PLOT_CODE. Set RANDOM_OBSERVER_CODE_LIST = 'all' or a list of codes to compare multiple codes.
RANDOM_OBSERVER_CODE_LIST = globals().get('RANDOM_OBSERVER_CODE_LIST', [PLOT_CODE])
RANDOM_OBSERVER_YMIN = 1e-2

RANDOM_OBSERVER_QUANTITIES = [
    ('DM_total_pc_cm3', r'DM [pc cm$^{-3}$]'),
    ('EM_ne2_total_pc_cm6', r'EM [pc cm$^{-6}$]'),
    ('EM_ne2_hot_pc_cm6', r'Hot EM [pc cm$^{-6}$]'),
]
LOS_TYPE_ORDER = ['toward_center_in_plane', 'anti_center_in_plane', 'vertical_to_disk']
LOS_TYPE_LABELS = {
    'toward_center_in_plane': 'Toward centre',
    'anti_center_in_plane': 'Anti-centre',
    'vertical_to_disk': 'Vertical',
}

def _available_random_observer_codes(outroot=OUTROOT):
    return sorted(p.parent.name for p in Path(outroot).glob('*/random_observer_special_los_DM_EM.csv') if is_production_output_name(p.parent.name))

def _select_random_observer_codes(selection=RANDOM_OBSERVER_CODE_LIST):
    available = _available_random_observer_codes()
    lookup = {str(code).lower(): code for code in available}
    if selection is None or selection == 'default':
        selected = [PLOT_CODE]
    elif selection == 'all':
        selected = available
    elif isinstance(selection, str):
        selected = [selection]
    else:
        selected = list(selection)
    selected = [lookup.get(str(code).lower(), code) for code in selected]
    selected = [code for code in selected if code in available]
    if not selected:
        print('No selected random-observer CSV files found. Available:', available)
    return selected

def _load_random_observer_rows(code):
    path = OUTROOT / code / 'random_observer_special_los_DM_EM.csv'
    rows = []
    with path.open(newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            parsed = {'code': code, 'los_type': row.get('los_type', '')}
            for key, val in row.items():
                if key in {'los_type'}:
                    continue
                try:
                    parsed[key] = float(val)
                except Exception:
                    parsed[key] = val
            rows.append(parsed)
    return rows

random_codes = _select_random_observer_codes()
random_rows = []
for code in random_codes:
    random_rows.extend(_load_random_observer_rows(code))

if not random_rows:
    print('No random-observer data to plot.')
else:
    n_rows = len(RANDOM_OBSERVER_QUANTITIES)
    n_cols = len(LOS_TYPE_ORDER)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.3 * n_cols, 3.75 * n_rows), constrained_layout=True, squeeze=False)
    code_colors = {code: plt.cm.tab10(i % 10) for i, code in enumerate(random_codes)}
    rng = np.random.default_rng(12345)

    for r, (quantity, ylabel) in enumerate(RANDOM_OBSERVER_QUANTITIES):
        row_vals = []
        for c, los_type in enumerate(LOS_TYPE_ORDER):
            ax = axes[r, c]
            positions = np.arange(1, len(random_codes) + 1)
            box_data = []
            used_positions = []
            used_labels = []
            for pos, code in zip(positions, random_codes):
                vals = np.asarray([
                    row.get(quantity, np.nan) for row in random_rows
                    if row.get('code') == code and row.get('los_type') == los_type
                ], dtype=float)
                vals = vals[np.isfinite(vals) & (vals > 0)]
                vals = vals[vals >= RANDOM_OBSERVER_YMIN]
                if vals.size == 0:
                    continue
                box_data.append(vals)
                used_positions.append(pos)
                used_labels.append(code_label_with_redshift(code))
                jitter = rng.normal(0.0, 0.035, size=vals.size)
                ax.scatter(np.full(vals.size, pos) + jitter, vals, s=13, alpha=0.45, color=code_colors[code], edgecolors='none')
                row_vals.append(vals)
            if box_data:
                bp = ax.boxplot(
                    box_data, positions=used_positions, widths=0.52,
                    showfliers=False, patch_artist=True,
                    medianprops={'color': 'black', 'lw': 1.4},
                    boxprops={'lw': 1.2}, whiskerprops={'lw': 1.1}, capprops={'lw': 1.1},
                )
                for patch, pos in zip(bp['boxes'], used_positions):
                    code = random_codes[int(pos) - 1]
                    patch.set_facecolor(code_colors[code])
                    patch.set_alpha(0.28)
            ax.set_yscale('log')
            ax.set_ylim(bottom=RANDOM_OBSERVER_YMIN)
            ax.set_xticks(used_positions)
            ax.set_xticklabels(used_labels, rotation=28, ha='right', fontsize=max(8, font_legend_small['size'] - 2))
            if r == 0:
                ax.set_title(LOS_TYPE_LABELS.get(los_type, los_type), fontdict=font_label)
            if c == 0:
                ax.set_ylabel(ylabel, fontdict=font_label)
            ax.tick_params(axis='y', labelsize=tick_size)
            ax.grid(True, which='both', axis='y', alpha=0.25)
        if row_vals:
            ymax = max(np.nanpercentile(np.concatenate(row_vals), 99.5), RANDOM_OBSERVER_YMIN * 10)
            for c in range(n_cols):
                axes[r, c].set_ylim(RANDOM_OBSERVER_YMIN, ymax * 1.4)
    fig.suptitle('Random observer special LOS', fontdict={'family': 'serif', 'size': 24}, y=1.01)
    suffix = 'all' if RANDOM_OBSERVER_CODE_LIST == 'all' else '_'.join(random_codes)
    save_figure(fig, f'random_observer_special_los_{safe_filename(suffix)}')
    plt.show()

    try:
        import pandas as pd
        display(pd.DataFrame(random_rows).head())
    except Exception:
        pass


## Random Observer Special LOS Across Selected Codes

Compare random-observer special LOS distributions for all codes in `PLOT_CODE_LIST`.


In [ ]:
# Compare random-observer special LOS results for every code selected by PLOT_CODE_LIST.
# If PLOT_CODE_LIST is 'default'/'all'/None, all available random-observer outputs are used.
RANDOM_OBSERVER_COMPARE_CODE_LIST = globals().get('PLOT_CODE_LIST', 'default')
RANDOM_OBSERVER_COMPARE_YMIN = 1e-2

COMPARE_RANDOM_OBSERVER_QUANTITIES = [
    ('DM_total_pc_cm3', r'DM [pc cm$^{-3}$]'),
    ('EM_ne2_total_pc_cm6', r'EM [pc cm$^{-6}$]'),
    ('EM_ne2_hot_pc_cm6', r'Hot EM [pc cm$^{-6}$]'),
]
COMPARE_LOS_TYPE_ORDER = ['toward_center_in_plane', 'anti_center_in_plane', 'vertical_to_disk']
COMPARE_LOS_TYPE_LABELS = {
    'toward_center_in_plane': 'Toward centre',
    'anti_center_in_plane': 'Anti-centre',
    'vertical_to_disk': 'Vertical',
}

def _available_random_observer_codes_compare(outroot=OUTROOT):
    return sorted(p.parent.name for p in Path(outroot).glob('*/random_observer_special_los_DM_EM.csv') if is_production_output_name(p.parent.name))

def _select_random_observer_compare_codes(selection=RANDOM_OBSERVER_COMPARE_CODE_LIST):
    available = _available_random_observer_codes_compare()
    lookup = {str(code).lower(): code for code in available}
    if selection is None or selection == 'default' or selection == 'all':
        selected = available
    elif isinstance(selection, str):
        selected = [selection]
    else:
        selected = list(selection)
    selected = [lookup.get(str(code).lower(), code) for code in selected]
    selected = [code for code in selected if code in available]
    if not selected:
        print('No selected random-observer CSV files found. Available:', available)
    return selected

def _load_random_observer_compare_rows(code):
    path = OUTROOT / code / 'random_observer_special_los_DM_EM.csv'
    rows = []
    with path.open(newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            parsed = {'code': code, 'los_type': row.get('los_type', '')}
            for key, val in row.items():
                if key == 'los_type':
                    continue
                try:
                    parsed[key] = float(val)
                except Exception:
                    parsed[key] = val
            rows.append(parsed)
    return rows

compare_codes = _select_random_observer_compare_codes()
compare_rows = []
for code in compare_codes:
    compare_rows.extend(_load_random_observer_compare_rows(code))

if not compare_rows:
    print('No random-observer data to compare.')
else:
    n_rows = len(COMPARE_RANDOM_OBSERVER_QUANTITIES)
    n_cols = len(COMPARE_LOS_TYPE_ORDER)
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(4.45 * n_cols, 3.9 * n_rows),
        constrained_layout=True,
        squeeze=False,
    )
    code_colors = {code: plt.cm.tab10(i % 10) for i, code in enumerate(compare_codes)}
    rng = np.random.default_rng(23456)

    for r, (quantity, ylabel) in enumerate(COMPARE_RANDOM_OBSERVER_QUANTITIES):
        row_vals = []
        for c, los_type in enumerate(COMPARE_LOS_TYPE_ORDER):
            ax = axes[r, c]
            used_positions = []
            used_labels = []
            box_data = []
            for pos, code in enumerate(compare_codes, start=1):
                vals = np.asarray([
                    row.get(quantity, np.nan) for row in compare_rows
                    if row.get('code') == code and row.get('los_type') == los_type
                ], dtype=float)
                vals = vals[np.isfinite(vals) & (vals >= RANDOM_OBSERVER_COMPARE_YMIN)]
                if vals.size == 0:
                    continue
                box_data.append(vals)
                used_positions.append(pos)
                used_labels.append(code_label_with_redshift(code))
                jitter = rng.normal(0.0, 0.035, size=vals.size)
                ax.scatter(
                    np.full(vals.size, pos) + jitter, vals,
                    s=11, alpha=0.38, color=code_colors[code], edgecolors='none', rasterized=True,
                )
                row_vals.append(vals)
            if box_data:
                bp = ax.boxplot(
                    box_data,
                    positions=used_positions,
                    widths=0.52,
                    showfliers=False,
                    patch_artist=True,
                    medianprops={'color': 'black', 'lw': 1.35},
                    boxprops={'lw': 1.1},
                    whiskerprops={'lw': 1.0},
                    capprops={'lw': 1.0},
                )
                for patch, pos in zip(bp['boxes'], used_positions):
                    code = compare_codes[int(pos) - 1]
                    patch.set_facecolor(code_colors[code])
                    patch.set_alpha(0.28)
            ax.set_yscale('log')
            ax.set_ylim(bottom=RANDOM_OBSERVER_COMPARE_YMIN)
            ax.set_xticks(used_positions)
            ax.set_xticklabels(used_labels, rotation=30, ha='right', fontsize=max(8, font_legend_small['size'] - 2))
            if r == 0:
                ax.set_title(COMPARE_LOS_TYPE_LABELS.get(los_type, los_type), fontdict=font_label)
            if c == 0:
                ax.set_ylabel(ylabel, fontdict=font_label)
            ax.tick_params(axis='y', labelsize=tick_size)
            ax.grid(True, which='both', axis='y', alpha=0.25)
        if row_vals:
            row_all = np.concatenate(row_vals)
            ymax = max(np.nanpercentile(row_all, 99.5), RANDOM_OBSERVER_COMPARE_YMIN * 20)
            for c in range(n_cols):
                axes[r, c].set_ylim(RANDOM_OBSERVER_COMPARE_YMIN, ymax * 1.5)
    #fig.suptitle('Random observer special LOS comparison across selected codes', fontdict={'family': 'serif', 'size': 24}, y=1.01)
    suffix = 'all' if RANDOM_OBSERVER_COMPARE_CODE_LIST in [None, 'default', 'all'] else '_'.join(compare_codes)
    save_figure(fig, f'random_observer_special_los_compare_{safe_filename(suffix)}')
    plt.show()


## Quick Dataset Structure Inspection

Use this lightweight panel to inspect each code's yt-visible data structure, units, particle types, and key gas/star fields before running expensive LOS calculations.


In [ ]:
from pathlib import Path
import sys
import importlib
import json
import csv
import numpy as np

# This notebook should be stored in the z0 directory.
ROOT = Path('/home/zhaozhang/local/AGORA_work/AGORA_Data/z0')
PIPELINE = ROOT / 'AGORA_parallel_DM_EM_projection_pipeline.py'

if not PIPELINE.exists():
    raise FileNotFoundError(f'Missing pipeline script: {PIPELINE}')

sys.path.insert(0, str(ROOT))
import AGORA_parallel_DM_EM_projection_pipeline as agora
agora = importlib.reload(agora)

# Quick structure inspection for AGORA z0 datasets.
# Edit this list if you only want a subset, e.g. ["Enzo", "G4Cal_Pablo"].

INSPECT_CODES = ["ARTI", "Enzo", "AREPO", "GADGET3", "GEAR", "CHANGA", "G4Cal_Pablo"]
INSPECT_MAX_FIELDS = 40
CODE_ALIASES = {
    'ARTI': 'ART-I',
    'ART-I': 'ART-I',
    'ART': 'ART-I',
    'ENZO': 'ENZO',
    'Enzo': 'ENZO',
    'AREPO': 'AREPO',
    'GADGET3': 'GADGET-3',
    'GADGET-3': 'GADGET-3',
    'GEAR': 'GEAR',
    'CHANGA': 'CHANGA',
    'G4Cal_Pablo': 'GADGET-4',
    'G4CAL_PABLO': 'GADGET-4',
    'G4Cal': 'GADGET-4',
}

FOLDER_ALIASES = {
    'ART-I': 'ARTI',
    'ENZO': 'Enzo',
    'AREPO': 'AREPO',
    'GADGET-3': 'GADGET3',
    'GADGET-4': 'G4Cal_Pablo',
    'GEAR': 'GEAR',
    'CHANGA': 'CHANGA',
    'G4Cal_Pablo': 'GADGET-4',
    'G4CAL_PABLO': 'GADGET-4',
    'G4Cal': 'GADGET-4',
}

def normalize_user_code(code):
    if code not in CODE_ALIASES:
        raise ValueError(f'Unknown code {code!r}. Use one of: {sorted(CODE_ALIASES)}')
    return CODE_ALIASES[code]

# This notebook should be stored in the z0 directory.
ROOT = Path('/home/zhaozhang/local/AGORA_work/AGORA_Data/z0')
PIPELINE = ROOT / 'AGORA_parallel_DM_EM_projection_pipeline.py'

if not PIPELINE.exists():
    raise FileNotFoundError(f'Missing pipeline script: {PIPELINE}')

sys.path.insert(0, str(ROOT))
import AGORA_parallel_DM_EM_projection_pipeline as agora
agora = importlib.reload(agora)


def candidate_snapshots_for_code(code, root=ROOT):
    normalized = normalize_user_code(code)
    folder = root / FOLDER_ALIASES[normalized]
    if not folder.exists():
        raise FileNotFoundError(f'Missing folder for {normalized}: {folder}')

    if normalized == 'ART-I':
        candidates = sorted(folder.glob('*.d'))
    elif normalized == 'ENZO':
        candidates = sorted(p for p in folder.glob('RD*/RD*') if p.is_file() and p.name == p.parent.name)
    elif normalized == 'AREPO':
        # Main AREPO snapshot is snap_###.hdf5. Exclude auxiliary files such as snap_###.hsml.hdf5.
        candidates = sorted(p for p in folder.glob('snap_*.hdf5') if '.hsml.' not in p.name and '.kdtree' not in p.name)
    elif normalized in {'GADGET-3', 'GADGET-4'}:
        # Multi-file Gadget snapshots should be opened from the .0.hdf5 member.
        candidates = (sorted(folder.glob('snapshot_*/*.0.hdf5')) + sorted(folder.glob('snapshot_*.hdf5')) + sorted(folder.glob('snap_*.hdf5')))
        candidates = [p for p in candidates if 'fof_subhalo' not in p.name]
    elif normalized == 'GEAR':
        candidates = sorted(p for p in (list(folder.glob('snapshot_*.hdf5')) + list(folder.glob('*.hdf5'))) if '.hsml.' not in p.name)
    elif normalized == 'CHANGA':
        # CHANGA/Tipsy main file is normally like ncal-IV.003524. Sidecars append names such as .HII or .massform.
        def is_changa_main_file(path):
            if not path.is_file() or not path.name.startswith('ncal-'):
                return False
            parts = path.name.split('.')
            return len(parts) == 2 and parts[-1].isdigit()
        preferred = sorted(p for p in folder.iterdir() if is_changa_main_file(p))
        fallback = sorted(
            p for p in folder.iterdir()
            if p.is_file() and not p.name.startswith('.') and p.name not in {'wget-log', 'robots.txt.tmp'}
            and not any(p.name.endswith(s) for s in ['.HII', '.massform', '.Metalsdot', '.ESNRate', '.kdtree'])
        )
        candidates = preferred or fallback
    else:
        candidates = []
    return normalized, folder, candidates

def discover_dataset(code, root=ROOT, index=0):
    normalized, folder, candidates = candidate_snapshots_for_code(code, root)
    if not candidates:
        raise FileNotFoundError(f'No candidate snapshot found for {normalized} in {folder}')
    snapshot = candidates[index]
    return {
        'input_code': code,
        'code': normalized,
        'folder': folder,
        'snapshot': snapshot,
        'candidate_count': len(candidates),
        'all_candidates': candidates,
    }

def _field_exists_for_inspection(ds, field):
    try:
        return field in ds.field_list or field in ds.derived_field_list
    except Exception:
        return False


def inspect_one_agora_dataset(dataset_code):
    selected = discover_dataset(dataset_code)
    code = selected["code"]
    snapshot = selected["snapshot"]
    norm_code = agora.normalize_code(code)

    print("\n" + "=" * 90)
    print(f"DATASET_CODE = {dataset_code}")
    print(f"normalized   = {norm_code}")
    print(f"snapshot     = {snapshot}")
    print(f"candidates   = {selected['candidate_count']}")

    ds = agora.load_dataset(str(snapshot), norm_code, "auto")
    code_cfg = agora.AGORA_CODE_CONFIG.get(norm_code, agora.default_config_for_unknown(norm_code, ds))

    print("\n[Units / cosmology]")
    for k, v in agora.dataset_unit_metadata(ds).items():
        print(f"  {k}: {v}")

    print("\n[yt dataset]")
    print("  dataset_type:", getattr(ds, "dataset_type", None))
    print("  particle_types:", getattr(ds, "particle_types", None))
    print("  particle_types_raw:", getattr(ds, "particle_types_raw", None))
    print("  domain_dimensions:", getattr(ds, "domain_dimensions", None))

    try:
        print("  field_count:", len(ds.field_list))
        print("  derived_field_count:", len(ds.derived_field_list))
    except Exception as exc:
        print("  field list load failed:", repr(exc))

    try:
        print("  particle_type_counts:", getattr(ds, "particle_type_counts", None))
    except Exception:
        pass

    print("\n[Chosen field types]")
    gas_ftype = agora.choose_ftype(ds, "auto", code_cfg.get("gas_types", ["gas"]), code_cfg.get("density_names", ["density"]))
    star_ftype = agora.choose_ftype(ds, "auto", code_cfg.get("star_types", []), ["particle_position_x", "particle_position", "Coordinates", "x"])
    print("  gas_ftype :", gas_ftype)
    print("  star_ftype:", star_ftype)
    if norm_code == "ENZO" and star_ftype == "all":
        print("  NOTE: Enzo 'all' is an aggregate particle container, not a clean stellar disk tracer.")

    print("\n[Key gas fields]")
    key_groups = {
        "density": code_cfg.get("density_names", []),
        "temperature": code_cfg.get("temperature_names", []),
        "electron": code_cfg.get("electron_names", []),
        "HII": code_cfg.get("hii_names", []),
        "HeII": code_cfg.get("heii_names", []),
        "HeIII": code_cfg.get("heiii_names", []),
        "mass": code_cfg.get("gas_mass_names", []),
        "smoothing_length": code_cfg.get("smoothing_length_names", []),
    }
    for label, names in key_groups.items():
        found = agora.first_existing_field(ds, code_cfg.get("gas_types", ["gas"]), names) if names else None
        print(f"  {label:16s}: {found}")

    print("\n[Key star fields]")
    for label, names in {
        "mass": code_cfg.get("star_mass_names", []),
        "position_x": ["particle_position_x", "x"],
        "position_vec": ["particle_position", "Coordinates"],
        "velocity_vec": code_cfg.get("velocity_vector_names", ["particle_velocity", "Velocities"]),
    }.items():
        found = agora.first_existing_field(ds, code_cfg.get("star_types", []), names) if names else None
        print(f"  {label:16s}: {found}")

    print("\n[Sample raw field names]")
    try:
        shown = 0
        for f in ds.field_list:
            if shown >= INSPECT_MAX_FIELDS:
                break
            print(" ", f)
            shown += 1
    except Exception as exc:
        print("  could not print field_list:", repr(exc))

    return ds


inspection_results = {}
for dataset_code in INSPECT_CODES:
    try:
        inspection_results[dataset_code] = inspect_one_agora_dataset(dataset_code)
    except Exception as exc:
        print("\n" + "=" * 90)
        print(f"FAILED inspecting {dataset_code}: {exc!r}")


## Stellar Mock-Observation Style Atlas

False-color stellar maps built from the projection arrays. This is not a JWST/NIRCam filter rendering; it is a compact visual diagnostic using stellar surface-density contrast.


In [ ]:
# Stellar mock-observation style atlas for all selected codes.
# This is a compact false-color diagnostic from stellar surface density maps,
# not a JWST/NIRCam filter synthesis.

if 'save_figure' not in globals() or 'load_json' not in globals():
    raise RuntimeError('Please run the Shared Results Plotting Utilities cell before this plotting panel.')

MOCK_OBS_CODE_LIST = globals().get('MOCK_OBS_CODE_LIST', globals().get('PLOT_CODE_LIST', 'default'))
MOCK_OBS_LIMIT_KPC = 5  # Force the mock-observation 2D maps to [-5, 5] kpc.
MOCK_OBS_FACE_EDGE_HEIGHT_RATIO = globals().get('MOCK_OBS_FACE_EDGE_HEIGHT_RATIO', 0.38)


def _mock_load_projection_products(outroot=OUTROOT):
    if '_load_projection_products' in globals():
        return _load_projection_products(outroot)
    products = {}
    for outdir in sorted(Path(outroot).glob('*')):
        if 'is_production_output_dir' in globals() and not is_production_output_dir(outdir):
            continue
        path = outdir / 'projection_maps_face_edge.npz'
        if path.exists():
            with np.load(path) as data:
                products[outdir.name] = {key: np.asarray(data[key]) for key in data.files}
    return products


def _mock_select_codes(products, selection):
    if '_code_selection_or_all' in globals():
        return _code_selection_or_all(products, selection)
    available = list(products)
    lookup = {str(code).lower(): code for code in available}
    if selection is None or selection == 'default' or selection == 'all':
        return available
    selected = [selection] if isinstance(selection, str) else list(selection)
    selected = [lookup.get(str(code).lower(), code) for code in selected]
    return [code for code in selected if code in products]


def _stellar_global_log_limits(products, codes, pmin=0.8, pmax=99.8):
    chunks = []
    for code in codes:
        data = products[code]
        for key in ['star_face_sigma', 'star_edge_sigma']:
            if key not in data:
                continue
            arr = np.asarray(data[key], dtype=float)
            pos = arr[np.isfinite(arr) & (arr > 0)]
            if pos.size:
                chunks.append(np.log10(pos))
    if not chunks:
        return None
    vals = np.concatenate(chunks)
    lo = np.nanpercentile(vals, pmin)
    hi = np.nanpercentile(vals, pmax)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return None
    return lo, hi


def stellar_sigma_to_mock_rgb(arr, log_limits):
    arr = np.asarray(arr, dtype=float)
    rgb = np.zeros(arr.shape + (3,), dtype=float)
    if log_limits is None:
        return rgb
    lo, hi = log_limits
    logv = np.log10(np.where(arr > 0, arr, np.nan))
    x = (logv - lo) / (hi - lo)
    x = np.clip(np.nan_to_num(x, nan=0.0, posinf=1.0, neginf=0.0), 0.0, 1.0)
    luma = np.arcsinh(6.0 * x) / np.arcsinh(6.0)
    core = np.clip((x - 0.70) / 0.30, 0.0, 1.0)
    rgb[..., 0] = np.clip(1.16 * luma + 0.14 * core, 0, 1)
    rgb[..., 1] = np.clip(0.82 * luma ** 1.05 + 0.24 * core, 0, 1)
    rgb[..., 2] = np.clip(0.42 * luma ** 1.28 + 0.26 * core, 0, 1)
    alpha = np.clip((x - 0.035) / 0.20, 0.0, 1.0)
    rgb *= alpha[..., None]
    return rgb


mock_products = _mock_load_projection_products()
mock_codes = _mock_select_codes(mock_products, MOCK_OBS_CODE_LIST)
mock_codes = [code for code in mock_codes if 'star_face_sigma' in mock_products[code] or 'star_edge_sigma' in mock_products[code]]

if not mock_codes:
    print('No selected stellar projection products found for mock-observation atlas.')
else:
    log_limits = _stellar_global_log_limits(mock_products, mock_codes)
    ncols = len(mock_codes)
    fig = plt.figure(figsize=(1.72 * ncols + 0.35, 2.55), constrained_layout=False)
    outer = fig.add_gridspec(
        1, ncols,
        left=0.045, right=0.995, bottom=0.16, top=0.80,
        wspace=0.06,
    )
    for col, code in enumerate(mock_codes):
        sub = outer[col].subgridspec(
            2, 1,
            height_ratios=[1.0, MOCK_OBS_FACE_EDGE_HEIGHT_RATIO],
            hspace=0.035,
        )
        data = mock_products[code]
        for row, (arr_key, ext_key, view) in enumerate([
            ('star_face_sigma', 'star_face_extent', 'face-on'),
            ('star_edge_sigma', 'star_edge_extent', 'edge-on'),
        ]):
            ax = fig.add_subplot(sub[row, 0])
            ax.set_facecolor('black')
            if arr_key in data and ext_key in data:
                rgb = stellar_sigma_to_mock_rgb(data[arr_key], log_limits)
                ax.imshow(rgb, origin='lower', extent=data[ext_key], interpolation='nearest')
            else:
                ax.text(0.5, 0.5, 'missing', transform=ax.transAxes, ha='center', va='center', color='white', fontsize=7)
            ax.set_xlim(-MOCK_OBS_LIMIT_KPC, MOCK_OBS_LIMIT_KPC)
            ax.set_ylim(-MOCK_OBS_LIMIT_KPC, MOCK_OBS_LIMIT_KPC)
            ax.set_box_aspect(1.0 if row == 0 else MOCK_OBS_FACE_EDGE_HEIGHT_RATIO)
            ax.tick_params(axis='both', labelsize=7, length=1.8, pad=1, colors='black')
            if row == 0:
                label = code_label_with_redshift(code) if 'code_label_with_redshift' in globals() else code
                ax.set_title(label, fontdict={'family': 'serif', 'size': 10}, pad=1.5)
                ax.tick_params(labelbottom=False)
            else:
                ax.set_xlabel('X [kpc]', fontdict={'family': 'serif', 'size': 8}, labelpad=0)
            if col == 0:
                ax.set_ylabel('Y [kpc]', fontdict={'family': 'serif', 'size': 8}, labelpad=0)
            else:
                ax.tick_params(labelleft=False)
            ax.text(
                0.03, 0.91, view, transform=ax.transAxes, ha='left', va='top',
                color='white', fontsize=6.5,
                bbox=dict(facecolor='black', alpha=0.35, edgecolor='none', pad=1.0),
            )
    fig.suptitle('Stellar mock-observation style projections', fontdict={'family': 'serif', 'size': 13}, y=0.965)
    save_figure(fig, 'stellar_mock_observation_style_atlas')
    plt.show()
